# WTI COT MM Nowcasting — Kalman Filter — 03 KF with Exogenous Features

We add the 5 selected features to the best-performing baseline state space structure from notebook 02 via two formulations:

| Model | How features enter | Params |
|---|---|---|
| **D — Local Level + Fixed Regression** | Observation equation intercept `d_t = X_t β` (fixed β) | 21 |
| **E — Dynamic Regression** | Time-varying `β_t` as additional state (random walk) | 18 |

**Key question:** Do features provide incremental signal beyond what the latent state captures?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../../')

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from statsmodels.tsa.statespace.mlemodel import MLEModel

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
from src.utils.io.read import PreprocessedDataReader
from src.preprocessing.base import FutureTicker
from src.settings import Settings

pdr = PreprocessedDataReader(Settings.historical.paths.PREPROCESSED_DATA_PATH)
dataset = pdr.read_dataset(ticker=FutureTicker.WTI)
dataset['tradeDate'] = pd.to_datetime(dataset['tradeDate'])
dataset.sort_values('tradeDate', inplace=True)
dataset.reset_index(drop=True, inplace=True)

print(f'Shape: {dataset.shape}')

In [ ]:
CONFIG_PATH = pathlib.Path('../../../cache/output/wti/mm/kf_config.json')
with open(CONFIG_PATH) as f:
    kf_config = json.load(f)

RESPONSES = kf_config['responses_raw']
FEATURES  = kf_config['selected_features']

print('Responses:', RESPONSES)
print('Features :', FEATURES)

---
## 1. Data Preparation

We need a joint clean matrix: rows where ALL responses AND ALL features are non-NaN.

In [ ]:
cols_needed = ['tradeDate'] + RESPONSES + FEATURES
df = dataset[cols_needed].dropna().reset_index(drop=True)

dates = df['tradeDate'].values
Y_raw = df[RESPONSES].values.astype(float)   # (T, 3)
X_raw = df[FEATURES].values.astype(float)    # (T, 5)

print(f'Joint clean matrix: {len(df)} observations '
      f'({df["tradeDate"].min().date()} → {df["tradeDate"].max().date()})')

In [ ]:
# Standardise responses and features separately
Y_mean, Y_std = Y_raw.mean(axis=0), Y_raw.std(axis=0)
X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0)

Y = (Y_raw - Y_mean) / Y_std
X = (X_raw - X_mean) / X_std

k_endog   = Y.shape[1]   # 3
n_features = X.shape[1]  # 5

print(f'k_endog={k_endog}, n_features={n_features}')
print(f'Y range (scaled): [{Y.min():.2f}, {Y.max():.2f}]')
print(f'X range (scaled): [{X.min():.2f}, {X.max():.2f}]')

---
## 2. Model Definitions

### Model D — Local Level + Fixed Exogenous Regression

```
Observation:  y_t = α_t + X_t β + ε_t    ε_t ~ N(0, H)
State:       α_{t+1} = α_t + η_t          η_t ~ N(0, Q)
```

- `α_t ∈ ℝ³` — latent positioning level  
- `β ∈ ℝ^{3×5}` — fixed regression coefficients (one row per response)  
- `H`, `Q` — diagonal 3×3 covariance matrices  
- **21 parameters** total: 3 (log_h) + 3 (log_q) + 15 (β)

In [ ]:
class LocalLevelWithFeatures(MLEModel):
    """
    Multivariate local level model with fixed exogenous regressors.

    y_t = α_t + X_t β + ε_t,   ε_t ~ N(0, H)  diagonal
    α_{t+1} = α_t + η_t,        η_t ~ N(0, Q)  diagonal

    Parameters (log-scale for variances):
        log_h  (k,)     — log obs noise std devs
        log_q  (k,)     — log state noise std devs
        beta   (k × p,) — flattened regression matrix (row = response)
    """

    def __init__(self, endog, exog):
        k = endog.shape[1]
        p = exog.shape[1]
        self.k_ = k
        self.p_ = p
        super().__init__(endog, k_states=k, k_posdef=k)

        self['design']     = np.eye(k)
        self['transition'] = np.eye(k)
        self['selection']  = np.eye(k)

        # Store exog — will be used to set obs_intercept in update()
        self._exog = np.array(exog)   # (T, p)

        self.initialize_approximate_diffuse()

    # ------------------------------------------------------------------ #
    @property
    def param_names(self):
        k, p = self.k_, self.p_
        names  = [f'log_h_{i}' for i in range(k)]
        names += [f'log_q_{i}' for i in range(k)]
        names += [f'beta_{i}_{j}' for i in range(k) for j in range(p)]
        return names

    @property
    def start_params(self):
        k, p = self.k_, self.p_
        log_std = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([log_std, log_std - 1.0, np.zeros(k * p)])

    # ------------------------------------------------------------------ #
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k, p = self.k_, self.p_

        h_std = np.exp(params[:k])
        q_std = np.exp(params[k:2*k])
        beta  = params[2*k:].reshape(k, p)   # (k, p)

        self['obs_cov']   = np.diag(h_std ** 2)
        self['state_cov'] = np.diag(q_std ** 2)

        # d_t = β x_t  →  obs_intercept shape: (k, nobs)
        self['obs_intercept'] = beta @ self._exog.T

### Model E — Dynamic Regression (Time-Varying Coefficients)

```
Observation:  y_t = Z_t β_t + ε_t    ε_t ~ N(0, H)
State:       β_{t+1} = β_t + η_t      η_t ~ N(0, Q)  (random walk)
```

- `β_t ∈ ℝ^{3×5}` — time-varying regression coefficients (stacked into a 15-dim state)  
- `Z_t = blockdiag(x_t, x_t, x_t)` — time-varying design matrix of shape `(3, 15)` per step  
- **18 parameters**: 3 (log_h diagonal) + 15 (log_q diagonal, one per coefficient)

This allows the market sensitivity to each feature to **drift over time**.

In [ ]:
class DynamicRegression(MLEModel):
    """
    Multivariate dynamic regression: time-varying β as state.

    y_t   = Z_t β_t + ε_t,   ε_t ~ N(0, H)  diagonal k×k
    β_{t+1} = β_t + η_t,      η_t ~ N(0, Q)  diagonal (k*p)×(k*p)

    State: β_t = [β_Net (p), β_Long (p), β_Short (p)]  dim = k*p
    Z_t   = block_diag(x_t', x_t', x_t')               shape (k, k*p)

    Parameters (log-scale):
        log_h  (k,)    — obs noise std devs
        log_q  (k*p,)  — state noise std devs (one per coefficient)
    """

    def __init__(self, endog, exog):
        k = endog.shape[1]
        p = exog.shape[1]
        T = endog.shape[0]
        self.k_ = k
        self.p_ = p
        k_states = k * p   # 15

        super().__init__(endog, k_states=k_states, k_posdef=k_states)

        # Build time-varying design matrix: (k, k*p, T)
        Z = np.zeros((k, k_states, T))
        for t in range(T):
            for i in range(k):
                Z[i, i*p:(i+1)*p, t] = exog[t]
        self['design'] = Z

        self['transition'] = np.eye(k_states)   # random walk
        self['selection']  = np.eye(k_states)

        self.initialize_approximate_diffuse()

    # ------------------------------------------------------------------ #
    @property
    def param_names(self):
        k, p = self.k_, self.p_
        names  = [f'log_h_{i}' for i in range(k)]
        names += [f'log_q_{i}_{j}' for i in range(k) for j in range(p)]
        return names

    @property
    def start_params(self):
        k, p = self.k_, self.p_
        log_std = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([log_std, np.full(k * p, -2.0)])

    # ------------------------------------------------------------------ #
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k, p = self.k_, self.p_

        h_std = np.exp(params[:k])
        q_std = np.exp(params[k:])

        self['obs_cov']   = np.diag(h_std ** 2)
        self['state_cov'] = np.diag(q_std ** 2)

---
## 3. Full-Sample Fit

In [ ]:
def fit_model(model_cls, endog, exog, start_params=None, maxiter=500):
    mod = model_cls(endog, exog)
    res = mod.fit(start_params=start_params, disp=False,
                  method='lbfgs', maxiter=maxiter)
    return mod, res

print('Fitting Model D — Local Level + Fixed Regression...')
mod_D, res_D = fit_model(LocalLevelWithFeatures, Y, X)
print(f'  LogLik: {res_D.llf:.2f}   AIC: {res_D.aic:.2f}   BIC: {res_D.bic:.2f}')

print('Fitting Model E — Dynamic Regression...')
mod_E, res_E = fit_model(DynamicRegression, Y, X, maxiter=600)
print(f'  LogLik: {res_E.llf:.2f}   AIC: {res_E.aic:.2f}   BIC: {res_E.bic:.2f}')

In [ ]:
# Load baseline ICs from notebook 02 for comparison
# (re-fit A and C on the same joint-clean dataset for a fair comparison)
from notebooks.cot_modeling.wti.wti_cot_mm_nowcast_kf_02_baseline_kf import (
    MultivariateLocalLevel, MultivariateLocalLinearTrend
)
# Note: if the import above fails, define the classes inline or copy them here
pass

In [ ]:
# Re-fit baseline models on the same (joint-clean) dataset for a fair IC comparison
class MultivariateLocalLevel(MLEModel):
    def __init__(self, endog, full_cov=False):
        k = endog.shape[1]
        self.k_ = k
        self.full_cov = full_cov
        super().__init__(endog, k_states=k, k_posdef=k)
        self['design'] = np.eye(k)
        self['transition'] = np.eye(k)
        self['selection'] = np.eye(k)
        self.initialize_approximate_diffuse()
    @property
    def param_names(self):
        k = self.k_
        names = [f'log_h_{i}' for i in range(k)] + [f'log_q_{i}' for i in range(k)]
        if self.full_cov:
            names += [f'h_cov_{i}{j}' for i in range(1, k) for j in range(i)]
        return names
    @property
    def start_params(self):
        log_std = np.log(np.std(self.endog, axis=0) + 1e-6)
        p = np.concatenate([log_std, log_std - 1.0])
        if self.full_cov:
            p = np.concatenate([p, np.zeros(self.k_ * (self.k_ - 1) // 2)])
        return p
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k = self.k_
        h_std = np.exp(params[:k])
        q_std = np.exp(params[k:2*k])
        if self.full_cov:
            L = np.diag(h_std)
            off = params[2*k:]
            idx = 0
            for i in range(1, k):
                for j in range(i):
                    L[i, j] = off[idx]; idx += 1
            H = L @ L.T
        else:
            H = np.diag(h_std ** 2)
        self['obs_cov'] = H
        self['state_cov'] = np.diag(q_std ** 2)

print('Re-fitting baseline A on joint-clean dataset...')
mod_A, res_A = MultivariateLocalLevel(Y, full_cov=False), None
mod_A_obj = MultivariateLocalLevel(Y, full_cov=False)
res_A = mod_A_obj.fit(disp=False, method='lbfgs', maxiter=400)
print(f'  AIC: {res_A.aic:.2f}   BIC: {res_A.bic:.2f}')

comparison = pd.DataFrame([
    {'Model': 'A: Local Level (diag)',           'n_params': len(res_A.params), 'AIC': res_A.aic, 'BIC': res_A.bic},
    {'Model': 'D: LL + Fixed Regression',        'n_params': len(res_D.params), 'AIC': res_D.aic, 'BIC': res_D.bic},
    {'Model': 'E: Dynamic Regression',           'n_params': len(res_E.params), 'AIC': res_E.aic, 'BIC': res_E.bic},
]).set_index('Model')
comparison.round(2)

### 3.1 Fitted Regression Coefficients (Model D)

Fixed-β coefficients tell us which features the MLE assigns predictive weight to, and the sign/magnitude relative to each response.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

k, p = k_endog, n_features
beta_hat = res_D.params[2*k:].reshape(k, p)

beta_df = pd.DataFrame(
    beta_hat,
    index=['Net', 'Long', 'Short'],
    columns=[f.replace('prior_report_', 'pr_').replace('prior_', 'pr_')
              .replace('F1_RolledPrice_', 'F1_').replace('_change', 'Δ')
              .replace('_rolling_20D_volatility', '_Vol20D')
              .replace('cumulative_5D_', '5D_')
              .replace('SyntheticF1MinusF2_RolledPrice', 'Synth')
             for f in FEATURES]
)

fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(beta_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Model D — Fitted Regression Coefficients β (standardised scale)')
ax.set_xlabel('Feature')
ax.set_ylabel('Response')
plt.tight_layout()
plt.show()

beta_df.round(4)

### 3.2 Time-Varying Coefficients (Model E)

The Kalman smoother gives $E[\beta_t \mid y_{1:T}]$ — we can visualise how each feature's coefficient drifts over time.

In [ ]:
# Extract smoothed β_t paths from Model E
smoothed_states = res_E.smoothed_state   # (k*p, T) = (15, T)
smoothed_std    = np.sqrt(res_E.smoothed_state_cov.diagonal())  # (k*p, T)

resp_labels = ['Net', 'Long', 'Short']
feat_short  = [
    'Price Δ', 'Vol 20D', 'AGG OI Δ', 'Spread Δ', 'Volume Δ'
]

fig, axes = plt.subplots(n_features, k_endog, figsize=(16, 3 * n_features), sharex=True)

for j, feat_label in enumerate(feat_short):
    for i, resp_label in enumerate(resp_labels):
        ax = axes[j, i]
        state_idx = i * n_features + j   # index in the 15-dim state
        mu  = smoothed_states[state_idx]
        sig = smoothed_std[state_idx]
        ax.plot(dates, mu, color='steelblue', linewidth=1)
        ax.fill_between(dates, mu - sig, mu + sig, alpha=0.25, color='steelblue')
        ax.axhline(0, color='grey', linestyle='--', linewidth=0.7)
        ax.set_title(f'{resp_label} ← {feat_label}', fontsize=9)
        if i == 0:
            ax.set_ylabel('β_t', fontsize=8)

axes[-1, 1].set_xlabel('Date')
plt.suptitle('Model E — Smoothed Time-Varying Coefficients β_t ± 1σ', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Walk-Forward Evaluation

In [ ]:
MIN_TRAIN   = 200
REFIT_EVERY = 26   # Model D: refit every 6 months
REFIT_EVERY_E = 52 # Model E: heavier, refit every year

print(f'OOS window: {len(Y) - MIN_TRAIN} observations')

In [ ]:
def walk_forward_with_features(model_cls, Y, X, Y_mean, Y_std,
                                min_train, refit_every, **model_kwargs):
    """
    Walk-forward 1-step-ahead evaluation for models with exogenous features.
    At step t, trains on Y[:t] / X[:t], forecasts y_{t+1} using x_{t+1}.
    """
    T = len(Y)
    preds = []
    current_params = None

    for t in range(min_train, T - 1):
        Y_train = Y[:t]
        X_train = X[:t]

        # Refit periodically
        if current_params is None or (t - min_train) % refit_every == 0:
            mod = model_cls(Y_train, X_train, **model_kwargs)
            try:
                res = mod.fit(disp=False, method='lbfgs', maxiter=300,
                              start_params=current_params)
                current_params = res.params
            except Exception:
                pass
        else:
            mod = model_cls(Y_train, X_train, **model_kwargs)
            res = mod.filter(current_params)

        # 1-step-ahead forecast
        filtered_state_last = res.filtered_state[:, -1]     # (k_states,)
        T_mat = mod['transition']                            # transition
        pred_state = T_mat @ filtered_state_last             # E[α_{t+1}|y_{1:t}]

        # For Model D: E[y_{t+1}] = Z pred_state + β x_{t+1}
        # For Model E: E[y_{t+1}] = Z_{t+1} pred_state
        # We construct Z using the NEXT period's features x_{t+1}
        x_next = X[t + 1]   # features available at t+1 for nowcasting
        Z_mat = mod['design']
        if Z_mat.ndim == 3:
            # Time-varying (Model E): rebuild Z for x_{t+1}
            k, k_st = mod.k_endog, mod.k_states
            p = mod.p_
            Z_next = np.zeros((k, k_st))
            for i in range(k):
                Z_next[i, i*p:(i+1)*p] = x_next
            forecast_obs = Z_next @ pred_state
        else:
            # Time-invariant (Model D): add obs_intercept
            k, p = mod.k_, mod.p_
            beta = current_params[2*k:].reshape(k, p)
            forecast_obs = Z_mat @ pred_state + beta @ x_next

        pred_orig = forecast_obs[:k_endog] * Y_std + Y_mean
        preds.append(pred_orig)

    preds = np.array(preds)           # (n_oos, 3)
    actuals = Y_raw[min_train + 1:]   # original scale

    return pd.DataFrame({
        'date':         dates[min_train + 1:],
        'pred_net':     preds[:, 0],
        'pred_long':    preds[:, 1],
        'pred_short':   preds[:, 2],
        'actual_net':   actuals[:, 0],
        'actual_long':  actuals[:, 1],
        'actual_short': actuals[:, 2],
    })

print('Running walk-forward for Model D (Local Level + Fixed Regression)...')
wf_D = walk_forward_with_features(
    LocalLevelWithFeatures, Y, X, Y_mean, Y_std, MIN_TRAIN, REFIT_EVERY
)

print('Running walk-forward for Model E (Dynamic Regression)...')
wf_E = walk_forward_with_features(
    DynamicRegression, Y, X, Y_mean, Y_std, MIN_TRAIN, REFIT_EVERY_E
)

print('Done.')

In [ ]:
def compute_metrics(wf_df):
    rows = []
    for r in ['net', 'long', 'short']:
        pred   = wf_df[f'pred_{r}'].values
        actual = wf_df[f'actual_{r}'].values
        mask   = ~(np.isnan(pred) | np.isnan(actual))
        rho, _ = stats.spearmanr(pred[mask], actual[mask])
        rmse   = np.sqrt(np.mean((pred[mask] - actual[mask]) ** 2))
        dir_acc = np.mean(np.sign(pred[mask]) == np.sign(actual[mask]))
        rows.append({'Response': r.capitalize(),
                     'Spearman ρ': rho, 'RMSE': rmse, 'Dir. Accuracy': dir_acc})
    return pd.DataFrame(rows).set_index('Response')

metrics_D = compute_metrics(wf_D)
metrics_E = compute_metrics(wf_E)

print('Model D — Local Level + Fixed Regression:')
print(metrics_D.round(4))
print('\nModel E — Dynamic Regression:')
print(metrics_E.round(4))

In [ ]:
# Load baseline walk-forward results (Model A) for comparison
OUT_DIR = pathlib.Path('../../../cache/output/wti/mm')
wf_A = pd.read_csv(OUT_DIR / 'kf_wf_baseline_A.csv', parse_dates=['date'])
metrics_A = compute_metrics(wf_A)

# Combined comparison
for metric in ['Spearman ρ', 'RMSE', 'Dir. Accuracy']:
    tab = pd.concat([
        metrics_A[[metric]].rename(columns={metric: 'A: LL-diag (baseline)'}),
        metrics_D[[metric]].rename(columns={metric: 'D: LL + Fixed Feat'}),
        metrics_E[[metric]].rename(columns={metric: 'E: Dynamic Reg'}),
    ], axis=1)
    print(f'\n--- {metric} ---')
    print(tab.round(4))

In [ ]:
# Rolling 52-week Spearman correlation (Net) — stability over time
window = 52

fig, ax = plt.subplots(figsize=(14, 4))
for wf, label, color in [
    (wf_A, 'A: LL-diag',          'grey'),
    (wf_D, 'D: LL + Fixed Feat',  'steelblue'),
    (wf_E, 'E: Dynamic Reg',      'darkorange'),
]:
    rolling_rho = [
        stats.spearmanr(
            wf['pred_net'].iloc[max(0, i-window):i],
            wf['actual_net'].iloc[max(0, i-window):i]
        ).statistic
        for i in range(window, len(wf))
    ]
    ax.plot(wf['date'].iloc[window:], rolling_rho, label=label, color=color, linewidth=1.2)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Rolling 52-week OOS Spearman ρ — Net Position Change')
ax.set_ylabel('Spearman ρ')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()

### 4.1 Residual Analysis — Model D

In [ ]:
# OOS residuals for Model D
for r, color in [('net', 'steelblue'), ('long', 'green'), ('short', 'tomato')]:
    resid = wf_D[f'actual_{r}'] - wf_D[f'pred_{r}']
    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    axes[0].plot(wf_D['date'], resid, color=color, linewidth=0.8)
    axes[0].axhline(0, color='grey', linestyle='--')
    axes[0].set_title(f'{r.capitalize()} — OOS Residuals')
    resid.hist(bins=40, ax=axes[1], color=color, alpha=0.7)
    axes[1].set_title('Residual Distribution')
    pd.plotting.autocorrelation_plot(resid.dropna(), ax=axes[2], color=color)
    axes[2].set_title('Residual ACF')
    axes[2].set_xlim(0, 30)
    plt.tight_layout()
    plt.show()

---
## 5. Feature Contribution Analysis (Model D)

For each time step, decompose the 1-step-ahead forecast into:
- **State component**: `Z α_{t|t-1}` (pure latent trend)
- **Feature component**: `X_t β` (exogenous contribution)

In [ ]:
# Full-sample decomposition using fitted Model D
mod_D_full = LocalLevelWithFeatures(Y, X)
mod_D_full.update(res_D.params)
res_D_full = mod_D_full.filter(res_D.params)

k, p = k_endog, n_features
beta_hat = res_D.params[2*k:].reshape(k, p)

# State contribution (latent trend)
state_contrib = res_D_full.filtered_state[:k].T * Y_std + Y_mean   # (T, 3)

# Feature contribution
feature_contrib = (X @ beta_hat.T) * Y_std + Y_mean  # (T, 3) — back-transform approximation
# Simpler: show raw feature contribution in standardised scale
feat_contrib_scaled = X @ beta_hat.T   # (T, 3)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for i, (ax, label) in enumerate(zip(axes, ['Net', 'Long', 'Short'])):
    ax.plot(dates, res_D_full.filtered_state[i] * Y_std[i] + Y_mean[i],
            label='State (latent trend)', color='steelblue', linewidth=1)
    ax.plot(dates, feat_contrib_scaled[:, i] * Y_std[i],
            label='Feature contribution', color='darkorange', linewidth=1, linestyle='--')
    ax.bar(dates, Y_raw[:, i], color='lightgrey', width=5, alpha=0.6, label='Observed Δ')
    ax.axhline(0, color='grey', linestyle=':', linewidth=0.7)
    ax.set_title(f'{label} — State vs Feature Contribution')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylabel('Contracts')
axes[-1].set_xlabel('Date')
plt.suptitle('Model D — Forecast Decomposition: State vs Features', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Summary

| Model | Formulation | Key result |
|---|---|---|
| A: Local Level (baseline) | No features | Captures autocorrelation only |
| D: LL + Fixed Regression | Fixed β, features in obs equation | Improves if β is stable |
| E: Dynamic Regression | Time-varying β as state | Adapts to regime changes in feature sensitivity |

**Design choice for Notebook 04:** Carry all three (A, D, E) into the full comparison alongside OLS and Ridge, evaluated on a common walk-forward window.

In [ ]:
# Save walk-forward results
wf_D.to_csv(OUT_DIR / 'kf_wf_model_D.csv', index=False)
wf_E.to_csv(OUT_DIR / 'kf_wf_model_E.csv', index=False)

# Save fitted beta for notebook 04
beta_dict = {
    'features': FEATURES,
    'responses': RESPONSES,
    'beta': beta_hat.tolist(),
}
with open(OUT_DIR / 'kf_model_D_beta.json', 'w') as f:
    json.dump(beta_dict, f, indent=2)

print('Results saved.')